In [20]:
from qiskit_aer import AerSimulator
from qiskit.circuit import *
from qiskit import QuantumCircuit, transpile

In [21]:
n = 3
while True:
    s = input("Digite uma string binária x de n bits: ").strip()
    if len(s) == n and all(bit in ('0', '1') for bit in s):
        break
    print("Entrada inválida! Digite exatamente n bits ('0' ou '1').")

In [22]:
def oraculo(s: str) -> QuantumCircuit:
    n = len(s)
    orac = QuantumCircuit(2*n, name="U_f")

    # inverte para little-endian do Qiskit
    s_rev = s[::-1]

    for i in range(n):
        orac.cx(i, n+i)

    if '1' in s_rev:
        indice = s_rev.index('1')
        for i in range(n):
            if s_rev[i] == '1':
                orac.cx(indice, n+i)

    return orac

circuito_oraculo = oraculo(s=s)
print("Circuito do Oráculo U_f:")
print(circuito_oraculo.draw('text'))

Circuito do Oráculo U_f:
                         
q_0: ──■─────────────────
       │                 
q_1: ──┼────■────────────
       │    │            
q_2: ──┼────┼────■────■──
     ┌─┴─┐  │    │    │  
q_3: ┤ X ├──┼────┼────┼──
     └───┘┌─┴─┐  │    │  
q_4: ─────┤ X ├──┼────┼──
          └───┘┌─┴─┐┌─┴─┐
q_5: ──────────┤ X ├┤ X ├
               └───┘└───┘


In [23]:
n = len(s)

qr1 = QuantumRegister(n, name='x_reg')
qr2 = QuantumRegister(n, name='fx_reg')
cr1 = ClassicalRegister(n, name='c_y')
cr2 = ClassicalRegister(n, name='c_fx')

circuito_quantico = QuantumCircuit(qr1, qr2, cr1, cr2)

circuito_quantico.h(qr1)
circuito_quantico.barrier()

circuito_quantico.compose(circuito_oraculo, qubits=[*qr1, *qr2], inplace=True)
circuito_quantico.barrier()

circuito_quantico.measure(qr2, cr2)
circuito_quantico.barrier()

circuito_quantico.h(qr1)
circuito_quantico.barrier()

circuito_quantico.measure(qr1, cr1)

print(circuito_quantico.draw('text'))

simulador = AerSimulator()
circuito_quant_compilado = transpile(circuito_quantico, simulador)
resultado = simulador.run(circuito_quant_compilado, shots=1024).result()
counts = resultado.get_counts()

y_counts={}

for saida, contagem in sorted(counts.items()):
    fx_medido, y_medido = saida.split(' ')
    y_counts[y_medido] = y_counts.get(y_medido, 0) + contagem
    produto_interno = sum(int(b1) * int(b2) for b1, b2 in zip(y_medido, s)) % 2
    print(f"  f(x) = {fx_medido} | y = {y_medido} -> {contagem:4d} ocorrências | y · x mod 2 = {produto_interno}")

          ┌───┐ ░                      ░           ░ ┌───┐ ░ ┌─┐      
 x_reg_0: ┤ H ├─░───■──────────────────░───────────░─┤ H ├─░─┤M├──────
          ├───┤ ░   │                  ░           ░ ├───┤ ░ └╥┘┌─┐   
 x_reg_1: ┤ H ├─░───┼────■─────────────░───────────░─┤ H ├─░──╫─┤M├───
          ├───┤ ░   │    │             ░           ░ ├───┤ ░  ║ └╥┘┌─┐
 x_reg_2: ┤ H ├─░───┼────┼────■────■───░───────────░─┤ H ├─░──╫──╫─┤M├
          └───┘ ░ ┌─┴─┐  │    │    │   ░ ┌─┐       ░ └───┘ ░  ║  ║ └╥┘
fx_reg_0: ──────░─┤ X ├──┼────┼────┼───░─┤M├───────░───────░──╫──╫──╫─
                ░ └───┘┌─┴─┐  │    │   ░ └╥┘┌─┐    ░       ░  ║  ║  ║ 
fx_reg_1: ──────░──────┤ X ├──┼────┼───░──╫─┤M├────░───────░──╫──╫──╫─
                ░      └───┘┌─┴─┐┌─┴─┐ ░  ║ └╥┘┌─┐ ░       ░  ║  ║  ║ 
fx_reg_2: ──────░───────────┤ X ├┤ X ├─░──╫──╫─┤M├─░───────░──╫──╫──╫─
                ░           └───┘└───┘ ░  ║  ║ └╥┘ ░       ░  ║  ║  ║ 
   c_y: 3/════════════════════════════════╬══╬══╬═════════════╩══╩══╩═
      

In [ ]:
# achar o s com os vetores ortogonais